In [1]:
import os
import sys
import pathlib as pl
sys.path.append(str(pl.Path().cwd().parent))
os.chdir(str(pl.Path().cwd().parent))
import json
from src.binance_data_collector import BinanceDataCollector
from src.data_preprocessor import DataPreprocessor


In [2]:
print(pl.Path.cwd())

/Users/franciscojara/Documents/lstm_work_crypto_volatility


In [3]:
with open('validation/healthy_dates.json', 'r') as f:
    config_dates = json.load(f)

In [4]:
config_dates

{'symbols': ['BTC-USD', 'ETH-USD'],
 'days_back': 365,
 'total_unique_dates': 366,
 'train': {'start': '2025-07-06', 'end': '2026-01-05', 'count': 184},
 'validation': {'start': '2026-01-06', 'end': '2026-04-06', 'count': 91},
 'test': {'start': '2026-04-07', 'end': '2026-07-06', 'count': 91}}

In [6]:
SYMBOLS = ['BTC-USD']
collector = BinanceDataCollector()
interval = '1m'
crypto_raw_data = collector.fetch_crypto_data(symbols = SYMBOLS, interval = interval, 
                                              days_back=config_dates['days_back'])

INFO:root:Fetching Binance data for 1 symbols...
INFO:root:Interval: 1m, Days back: 365
INFO:root:Fetching 1m data for BTCUSDT...
INFO:root:Fetched 1000 records for BTC-USD
INFO:root:Fetching 1m data for BTCUSDT...
INFO:root:Fetched 1000 records for BTC-USD
INFO:root:Fetching 1m data for BTCUSDT...
INFO:root:Fetched 1000 records for BTC-USD
INFO:root:Fetching 1m data for BTCUSDT...
INFO:root:Fetched 1000 records for BTC-USD
INFO:root:Fetching 1m data for BTCUSDT...
INFO:root:Fetched 1000 records for BTC-USD
INFO:root:Fetching 1m data for BTCUSDT...
INFO:root:Fetched 1000 records for BTC-USD
INFO:root:Fetching 1m data for BTCUSDT...
INFO:root:Fetched 1000 records for BTC-USD
INFO:root:Fetching 1m data for BTCUSDT...
INFO:root:Fetched 1000 records for BTC-USD
INFO:root:Fetching 1m data for BTCUSDT...
INFO:root:Fetched 1000 records for BTC-USD
INFO:root:Fetching 1m data for BTCUSDT...
INFO:root:Fetched 1000 records for BTC-USD
INFO:root:Fetching 1m data for BTCUSDT...
INFO:root:Fetched 10

In [8]:
preprocessor = DataPreprocessor()
processed_data = preprocessor.prepare_features(crypto_raw_data)
processed_data.columns

INFO:root:Starting feature preparation...


Calculating volatility from OHLC data...


INFO:root:Added volatility features
INFO:root:Calculating technical indicators...
INFO:root:Added technical indicators
INFO:root:Using target: Volatility_20d
INFO:root:Creating lagged features...
INFO:root:Creating momentum features...
INFO:root:Creating OHLC ratios...
INFO:root:Cleaning data...
INFO:root:Removed 14 rows with NaN/inf values
INFO:root:Feature preparation complete. Total features: 55
INFO:root:Dataset shape: (525586, 58)
INFO:root:Features: 55
INFO:root:Target: Volatility_20d


Index(['Date', 'Symbol', 'Open', 'High', 'Low', 'Close', 'Volume',
       'Quote_volume', 'Count', 'Taker_buy_volume', 'Taker_buy_quote_volume',
       'Returns', 'Volatility_5d', 'Volatility_10d', 'Volatility_20d',
       'Volatility_30d', 'Parkinson_Vol', 'GK_Vol', 'True_Range', 'ATR_14',
       'RSI', 'MA_5', 'Price_MA_5_Ratio', 'MA_10', 'Price_MA_10_Ratio',
       'MA_20', 'Price_MA_20_Ratio', 'MA_50', 'Price_MA_50_Ratio', 'BB_Upper',
       'BB_Lower', 'BB_Position', 'Close_lag_1', 'Returns_lag_1',
       'Volatility_20d_lag_1', 'Volume_lag_1', 'Close_lag_2', 'Returns_lag_2',
       'Volatility_20d_lag_2', 'Volume_lag_2', 'Close_lag_3', 'Returns_lag_3',
       'Volatility_20d_lag_3', 'Volume_lag_3', 'Close_lag_5', 'Returns_lag_5',
       'Volatility_20d_lag_5', 'Volume_lag_5', 'Close_lag_10',
       'Returns_lag_10', 'Volatility_20d_lag_10', 'Volume_lag_10',
       'Momentum_3d', 'Momentum_5d', 'Momentum_10d', 'Momentum_14d',
       'HL_Ratio', 'OC_Ratio'],
      dtype='object')

In [9]:
INPUT_STUDY_FEATURES = ['Returns', 'Volume', 'Count']
processed_data = preprocessor.scale_features(processed_data.set_index(['Date', 'Symbol']), scaler = 'minmax')
processed_data

,,Open,High,Low,Close,Volume,Quote_volume,Count,Taker_buy_volume,Taker_buy_quote_volume,Returns,...,Close_lag_10,Returns_lag_10,Volatility_20d_lag_10,Volume_lag_10,Momentum_3d,Momentum_5d,Momentum_10d,Momentum_14d,HL_Ratio,OC_Ratio
Date,Symbol,,,,,,,,,,,,,,,,,,,,,
2025-07-09 21:52:00,BTC-USD,0.778020,0.776289,0.777988,0.777326,0.004303,0.005297,0.009335,0.001975,0.001824,0.641980,...,0.778354,0.650174,0.009010,0.001070,0.587096,0.627930,0.665445,0.588535,7.541209e-03,0.009158
2025-07-09 21:53:00,BTC-USD,0.777326,0.775606,0.778061,0.777339,0.001794,0.002219,0.003065,0.002957,0.002730,0.648548,...,0.778221,0.647192,0.009524,0.002502,0.591412,0.630190,0.666102,0.589102,2.834240e-04,0.000168
2025-07-09 21:54:00,BTC-USD,0.777339,0.776435,0.778091,0.777910,0.005945,0.007314,0.010490,0.006548,0.006043,0.653740,...,0.779385,0.659245,0.021627,0.003949,0.597229,0.638301,0.663436,0.590648,7.954087e-03,0.007536
2025-07-09 21:55:00,BTC-USD,0.777910,0.776179,0.778481,0.777729,0.000757,0.000948,0.002009,0.000601,0.000557,0.646750,...,0.779458,0.649106,0.020008,0.002767,0.601325,0.640173,0.662295,0.588686,1.734552e-03,0.002380
2025-07-09 21:56:00,BTC-USD,0.777730,0.775998,0.778481,0.777730,0.001146,0.001425,0.000807,0.000981,0.000908,0.648429,...,0.779458,0.648429,0.018902,0.003770,0.601223,0.641025,0.662295,0.587979,1.609430e-07,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-07-09 21:33:00,BTC-USD,0.078946,0.075701,0.080043,0.078890,0.000416,0.000292,0.001225,0.000204,0.000107,0.647513,...,0.079288,0.644035,0.015573,0.000836,0.599113,0.645184,0.666931,0.583376,9.475612e-04,0.001302
2026-07-09 21:34:00,BTC-USD,0.078890,0.075645,0.079932,0.078779,0.000899,0.000630,0.002112,0.000062,0.000032,0.646624,...,0.079139,0.645992,0.015195,0.000771,0.595770,0.642744,0.667237,0.582646,1.866943e-03,0.002560
2026-07-09 21:35:00,BTC-USD,0.078779,0.075544,0.079791,0.078790,0.002504,0.001753,0.003450,0.001012,0.000532,0.648595,...,0.079081,0.647497,0.014241,0.002724,0.595914,0.642319,0.667770,0.583341,2.548953e-03,0.000237


In [10]:
config_dates

{'symbols': ['BTC-USD', 'ETH-USD'],
 'days_back': 365,
 'total_unique_dates': 366,
 'train': {'start': '2025-07-06', 'end': '2026-01-05', 'count': 184},
 'validation': {'start': '2026-01-06', 'end': '2026-04-06', 'count': 91},
 'test': {'start': '2026-04-07', 'end': '2026-07-06', 'count': 91}}

In [11]:
# Build training and validation splits here from the API-fetched dataframe.
lstm_data = preprocessor.prepare_lstm_data(processed_data.reset_index(), symbols = SYMBOLS,
                                           sequence_length = 5,
                                           target_col = 'Returns',
                                           prediction_horizon=1,
                                           date_splits =  {
                                               'train': config_dates['train'],
                                               'validation': config_dates['validation'],
                                               'test': config_dates['test']
                                           },
                                           input_study_features = INPUT_STUDY_FEATURES
                                           )
    

INFO:root: LSTM matrix transformation complete
INFO:root:   X_train: (257884, 5, 3), y_train: (257884,)
INFO:root:   train dates:      2025-07-09 to 2026-01-05
INFO:root:   X_val:   (129601, 5, 3), y_val:   (129601,)
INFO:root:   val dates:        2026-01-06 to 2026-04-06
INFO:root:   X_test:  (129601, 5, 3), y_test:  (129601,)
INFO:root:   test dates:       2026-04-07 to 2026-07-06


In [ ]:
print(lstm_data.keys())
lstm_data['dates_train']

dict_keys(['X_train', 'y_train', 'dates_train', 'X_val', 'y_val', 'dates_val', 'X_test', 'y_test', 'dates_test', 'feature_columns', 'target_col', 'sequence_length', 'prediction_horizon', 'symbols'])


array(['2025-07-09T21:57:00.000000000', '2025-07-09T21:58:00.000000000',
       '2025-07-09T21:59:00.000000000', ...,
       '2026-01-04T23:58:00.000000000', '2026-01-04T23:59:00.000000000',
       '2026-01-05T00:00:00.000000000'], dtype='datetime64[ns]')

In [16]:
lstm_data['y_train']

array([0.6444022 , 0.64967092, 0.63348803, ..., 0.65356192, 0.65647156,
       0.6587826 ])

In [20]:
lstm_data['X_train']

array([[[0.64197965, 0.0043027 , 0.00933496],
        [0.648548  , 0.00179381, 0.00306472],
        [0.65373998, 0.00594537, 0.01048951],
        [0.6467498 , 0.00075715, 0.00200873],
        [0.64842945, 0.00114636, 0.00080725]],

       [[0.648548  , 0.00179381, 0.00306472],
        [0.65373998, 0.00594537, 0.01048951],
        [0.6467498 , 0.00075715, 0.00200873],
        [0.64842945, 0.00114636, 0.00080725],
        [0.6444022 , 0.00184823, 0.00343549]],

       [[0.65373998, 0.00594537, 0.01048951],
        [0.6467498 , 0.00075715, 0.00200873],
        [0.64842945, 0.00114636, 0.00080725],
        [0.6444022 , 0.00184823, 0.00343549],
        [0.64967092, 0.00380341, 0.01002487]],

       ...,

       [[0.64767248, 0.00132155, 0.00362792],
        [0.65080248, 0.00274569, 0.00486695],
        [0.64332128, 0.00500196, 0.00487164],
        [0.65146277, 0.01255966, 0.00326184],
        [0.65366125, 0.00193892, 0.00529403]],

       [[0.65080248, 0.00274569, 0.00486695],
        [0.64

In [19]:
processed_data[['Returns', 'Volume', 'Count']].head(10)

,,Returns,Volume,Count
Date,Symbol,,,
2025-07-09 21:52:00,BTC-USD,0.641980,0.004303,0.009335
2025-07-09 21:53:00,BTC-USD,0.648548,0.001794,0.003065
2025-07-09 21:54:00,BTC-USD,0.653740,0.005945,0.010490
2025-07-09 21:55:00,BTC-USD,0.646750,0.000757,0.002009
2025-07-09 21:56:00,BTC-USD,0.648429,0.001146,0.000807
2025-07-09 21:57:00,BTC-USD,0.644402,0.001848,0.003435
2025-07-09 21:58:00,BTC-USD,0.649671,0.003803,0.010025
2025-07-09 21:59:00,BTC-USD,0.633488,0.005499,0.010917
2025-07-09 22:00:00,BTC-USD,0.666204,0.004648,0.014254
